# Augmentation by Albumentation

The core tenet of data centric machine learning is prioritizing high-quality, diverse, and relevant data for training models, recognizing its crucial role in model performance and generalization.

One technique that can help you with this is _augmentation_. Augmentation is the process of expanding a dataset by applying transformations like rotation, scaling, or noise to the data. This can help increase diversity in datasets, which in turn helps with generalization.

Augmentation is frequently associated with image data, but not exclusive to it. For instance, chemical applications of ML, augmentations such as conformer generation (different spatial arrangements of atoms in molecules) are used frequently. We will stick to images today.

At a first glance, augmentation looks very simple. It's really just a few basic transformations on an image. However, there are many pitfalls once masks, labels, boxes etc. get involved. This is why you're better off to use a dedicated augmentation library instead of building your own augmentations from scratch. One such library is [Albumentations](https://albumentations.ai).

In their own words,

> Albumentations is a fast and flexible image augmentation library. The library is widely used in industry, deep learning research, machine learning competitions, and open source projects. Albumentations is written in Python, and it is licensed under the MIT license. The source code is available at <https://github.com/albumentations-team/albumentations>.

You can install `albumentations` from `pip` like any regular python package. We've already taken care of this in the conda environment.

> **Note on versions:** Albumentations 2.0.8 (June 2025) is the last release of the original, MIT-licensed library - its GitHub repository is archived. Development continues as [AlbumentationsX](https://github.com/albumentations-team/AlbumentationsX), which has almost the same API but a different license (AGPL-3.0 or commercial). The documentation on albumentations.ai now describes AlbumentationsX. We use version 2.0.8 in this lab; everything shown here works the same in both.

## Import the required libraries

To start our augmentation adventure, we of course need the `albumentations` library. On top of this, we also require a library to read images from the disk. Here, we'll use [`pillow`](https://pillow.readthedocs.io/).

In [ ]:
import albumentations as A
from PIL import Image
import numpy as np

## Augmentation pipelines

Albumentations is built around the concept of an "augmentation pipeline". If you have ever worked with [PyTorch transforms](https://docs.pytorch.org/tutorials/beginner/basics/transforms_tutorial.html) or [`torchvision.transforms`](https://docs.pytorch.org/vision/stable/transforms.html), the syntax should remind you of something.

To define an augmentation pipeline, you need to create an instance of the `Compose` class. `Compose` takes a list of transformations (or "augmentations") to apply.

In [ ]:
transform = A.Compose([
    A.RandomCrop(width=256, height=256),  # randomly crop the image to 256x256
    A.HorizontalFlip(p=0.5), # horizontally flip 50% of the images
    A.RandomBrightnessContrast(p=0.2), # randomly adjust brightness and contrast in 20% of the images
])

This pipeline performs three augmentations. It randomly crops the image to 256x256 (the image must be at least that large), flips 50% of the images horizontally, and randomly changes brightness and contrast in 20% of the images. You can find a list of all available augmentations [in the docs](https://albumentations.ai/docs/reference/supported-targets-by-transform/).

The parameters of the individual augmentations depend on the type of augmentation, but there is a recurring one: `p`. `p` is a special parameter that is supported by almost all augmentations. It controls the probability of applying the augmentation. `p=0.3` indicates a 30% probability of applying the augmentation.

Here is a visualized version of the augmentation pipeline, courtesy of the [albumentations documentation](https://albumentations.ai/docs/2-core-concepts/pipelines/).

![image.png](../imgs/augmentation_pipeline_visualized.jpg)

## Read and transform images

Once you have created your augmentation pipeline, all that remains is reading and transforming your images.

In [ ]:
import matplotlib.pyplot as plt

# Read the image with PIL
image = np.array(Image.open("../imgs/cat.jpg"))

plt.imshow(image)
plt.axis('off')
plt.show()


Then, transforming it is just as easy!

In [ ]:
transformed = transform(image=image)
transformed_image = transformed["image"]
plt.imshow(transformed_image)
plt.axis('off')
plt.show()

Run the cell above a few times: since the augmentations are random, you get a different result every time.

And that's it!

Here are a few more examples you can look at if you want to learn more about albumentations:

- [Defining a simple augmentation pipeline for image augmentation](https://albumentations.ai/docs/examples/example/)
- [Weather augmentations in Albumentations](https://albumentations.ai/docs/examples/example-weather-transforms/)
- [Cool augmentation examples on a diverse set of images from various real-world tasks](https://albumentations.ai/docs/examples/showcase/)

## Augmentation at scale

When working with large amounts of data, you'll want to persist the augmentations and not compute them on the fly, every time you use the dataset.
In the last section on Albumentations, we will combine augmentation with what we have learnt about data versioning in the previous part.

We provide a simple python script to run Albumentations from the command line: [`lab06/apply_augmentation.py`](../apply_augmentation.py). It applies one augmentation to every image in a directory and saves each image the augmentation was actually applied to as a **new file** next to the original, e.g. `image_00001.jpg` → `image_00001_HorizontalFlip.jpg`. Take a look at it:

In [ ]:
!cat ../apply_augmentation.py

To run this script, use a command like this:

```shell
python apply_augmentation.py <input_dir> <output_dir> --augmentation <AugmentationName> --augmentation_params param1=value1 param2=value2 ...
```

`<AugmentationName>` is the name of an Albumentations class (e.g. `HorizontalFlip`), and the parameters are passed to it (e.g. `p=0.1` or `width=200`).

It is a bit limited in as far as it can only apply a single augmentation in each run, but you can of course re-run it on the already augmented data and "emulate" `A.Compose` in this way.

---

## A little bit more on data version control

Use the script to augment the `102flowers` data in the DVC repository you created in the [DVC part](../dvc.md#dvc-hands-on)! Run the following commands **in a terminal** (not in this notebook), from the **root of your DVC repository**, with the `mlops-lab-06` environment activated. Replace `<lab06>` with the path to the `lab06` folder of this course repository.

Input and output directory are both `data/102flowers`: the augmented images are added to the dataset as new files, the original images stay untouched.

Flip about 10% of the images horizontally:

```shell
python <lab06>/apply_augmentation.py data/102flowers data/102flowers --augmentation HorizontalFlip --augmentation_params p=0.1
```

Randomly crop about 10% of the images to 200x200 pixels:

```shell
python <lab06>/apply_augmentation.py data/102flowers data/102flowers --augmentation RandomCrop --augmentation_params width=200 height=200 p=0.1
```

Randomly change brightness and contrast of about 10% of the images:

```shell
python <lab06>/apply_augmentation.py data/102flowers data/102flowers --augmentation RandomBrightnessContrast --augmentation_params p=0.1
```

Most transformations are supported. **Always set the `p` parameter**: many augmentations default to `p=1` or `p=0.5`, which would add thousands of images to the dataset - and every run also augments the images added by previous runs. Each run processes all ~8000 images and takes a minute or two.

Once you are happy with your augmentations, check what has changed compared to the last commit:

```shell
dvc status   # data/102flowers is reported as modified
dvc diff     # lists the added files
```

Follow this up by the actual commit - first the data (DVC), then the pointer (Git) - and upload the new data to the remote:

```shell
dvc add data/102flowers
git add data/102flowers.dvc
git commit -m "Add augmentations to the input images"
dvc push
git push
```

Note that the data version is tied to the `*.dvc` file! Look at `data/102flowers.dvc` again: `md5`, `size` and `nfiles` have changed.

When you want to check out an old version of the data, you first check out the `.dvc` file from the Git commit that corresponds to this data version, and only then check out the data using `dvc checkout`. For the data before the augmentation, that is the commit before the most recent one, `HEAD~1`:

```shell
git checkout HEAD~1 -- data/102flowers.dvc   # restore the old pointer
dvc checkout                                 # restore the matching data
ls data/102flowers | wc -l                   # 8189 images again
```

To go back to the latest version, run `git checkout HEAD -- data/102flowers.dvc` followed by `dvc checkout`.